# 추론 및 제출

**이 노트북이 제출물이다.** Kaggle 이 숨은 테스트셋으로 다시 실행하고
`/kaggle/working/submission.csv` 를 채점한다

학습과 다른 점

- 픽셀 캐시를 쓸 수 없다. 숨은 테스트 스터디는 캐시에 없다
- DICOM 에서 실시간 전처리한다. 그래서 코덱 휠이 필요하다
- 9시간 런타임 제한이 여기서 걸린다

제출이 조용히 깨지는 사례들 (디스커션 735854, 742050)

- P100 은 사전설치 PyTorch 와 비호환인데 `cuda.is_available()` 는 True 다
- 코덱이 없는데 채점 실행은 인터넷이 꺼져 있다
- `dataset_sources` / `model_sources` 누락 시 조용히 드롭된다


In [ ]:
import json, os, subprocess, sys, time, traceback
from pathlib import Path

T0 = time.time()
diag = {}

for _c in ['/kaggle/input/datasets/hyejinkwak/knee-src', '/kaggle/input/knee-src']:
    if os.path.isdir(_c):
        sys.path.insert(0, _c); print('src:', _c); break
else:
    raise FileNotFoundError('knee-src 없음')

import paths, config
config.verify_labels()
L = list(config.LABELS)
print(paths.describe())
print('레이블 순서 검증 통과')


## 1. 제출 파일을 먼저 쓴다

뒤에서 무슨 일이 생겨도 유효한 파일이 남아 있어야 한다
모델 예측이 준비되면 덮어쓴다


In [ ]:
import numpy as np, pandas as pd

test = pd.read_csv(paths.TEST_CSV)
print(f'test studies: {len(test):,}')
diag['n_test'] = int(len(test))

# 공개 레이블 테이블의 사전확률 (stevenleehans v4_blend, CC0)
PRIOR = {'ACL': 0.2363, 'MCL': 0.1808, 'Medial Meniscus': 0.4605,
         'Lateral Meniscus': 0.2263, 'Medial OA': 0.4070, 'Lateral OA': 0.3263,
         'PF OA': 0.4831, 'Effusion': 0.5870, 'Synovitis': 0.3402,
         "Baker's": 0.3574, 'Contusion': 0.2636, 'Fracture': 0.1855}
assert set(PRIOR) == set(L)

studies = test[config.ID_COL].tolist()
fallback = np.tile([PRIOR[c] for c in L], (len(studies), 1))
config.make_submission(studies, fallback, path=paths.SUBMISSION_CSV)
print('fallback submission 작성 완료')


## 2. 코덱과 모델 준비


In [ ]:
wheels = paths.find_input('knee-dicom-wheels')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                '--find-links', str(wheels),
                'pylibjpeg', 'pylibjpeg-libjpeg', 'pylibjpeg-openjpeg', 'python-gdcm'],
               check=True)
import pydicom, torch
import cache as cache_mod, dicom_io, dataset as ds, model as M, infer
print('pydicom', pydicom.__version__, '| torch', torch.__version__)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    cc = torch.cuda.get_device_capability(0)
    print('gpu', torch.cuda.get_device_name(0), cc)
    assert cc >= (7, 0), 'P100 은 사전설치 PyTorch 와 호환되지 않는다'
diag['device'] = str(device)


In [ ]:
# 체크포인트 - 여러 개면 rank 평균으로 합친다
wdir = paths.find_input('knee-weights')
ckpts = sorted(Path(wdir).glob('**/*.pt'))
assert ckpts, f'{wdir} 에 .pt 가 없다'
print(f'체크포인트 {len(ckpts)}개')

nets, metas = [], []
for c in ckpts:
    net, ck = infer.load_model(c, device)
    nets.append(net); metas.append(ck)
    print(f'  {c.name}  val_auc {ck.get("val_auc", float("nan")):.4f}  '
          f'epoch {ck.get("epoch")}  plan {ck["train_cfg"]["plan"]}')

PLAN = metas[0]['train_cfg']['plan']
assert all(m['train_cfg']['plan'] == PLAN for m in metas), 'plan 이 서로 다르다'
diag['n_models'] = len(nets)
diag['plan'] = PLAN
diag['val_auc'] = [round(float(m.get('val_auc', 0)), 4) for m in metas]


## 3. 추론

숨은 테스트셋 크기를 모른다. 진행 상황과 속도를 계속 찍어 9시간 제한을 감시한다


In [ ]:
icfg = infer.InferConfig(plan=PLAN, batch_size=8, amp=True)
ccfg = cache_mod.CacheConfig()
print('slots:', ccfg.slots)
print('출력 칸:', len(ds.resolve_plan(PLAN)))

t = time.time()
probs = infer.predict_studies(nets, studies, icfg, ccfg, device,
                              train=False, log_every=100)
el = time.time() - t
print(f'\n추론 {el/60:.1f}분  ({el/max(len(studies),1)*1000:.0f} ms/study)')
diag['infer_min'] = round(el/60, 2)
diag['ms_per_study'] = round(el/max(len(studies),1)*1000, 1)
print(f'9시간 제한 기준 최대 {9*3600/(el/max(len(studies),1)):,.0f} 스터디까지 가능')


## 4. 제출 파일 작성과 검증


In [ ]:
sub = infer.write_submission(studies, probs, paths.SUBMISSION_CSV)
print()
print(sub.head().to_string())
print()
print('예측 분포 (레이블별 평균)')
print(sub[L].mean().round(4).to_string())

# 모든 값이 같으면 모델이 아무것도 안 배운 것이다
spread = float(sub[L].std().mean())
print(f'\n레이블별 표준편차 평균 {spread:.4f}')
assert spread > 1e-6, '예측이 상수다 - 모델이나 입력을 확인할 것'
diag['pred_std'] = round(spread, 4)


In [ ]:
diag['total_min'] = round((time.time()-T0)/60, 2)
json.dump(diag, open('/kaggle/working/submit_diag.json','w'), indent=2, ensure_ascii=False)
print(json.dumps(diag, indent=2, ensure_ascii=False))
print()
print('output:', sorted(os.listdir('/kaggle/working')))
